#1.Problem Statement

Identify the Electric Vehicle Demand, based on the available data.

#2.Exploratory Data Analysis

##2.1 Loading a dataset

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("/Volumes/electric_vehicle/ev_sales/ev_sales_data_raw/train.csv")

##2.2 Inspecting the dataset

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df.describe()

###2.2.1 Checking for missing values

In [0]:
df.isna().sum()

###2.2.2 Slicing Data into groups

In [0]:
df.loc[0:10,["City_Type","Will_Buy_EV"]] #.loc screens by label

In [0]:
df.iloc[0:10, [9, 14]] #.iloc screens by index position

In [0]:
df.iloc[0:10, 9:14]

In [0]:
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

How many of these customers travel 30 km or more everyday?

In [0]:
thirty_km_commuters = df[df.Daily_Commute_km == 30]

In [0]:
thirty_km_commuters.shape

#### 2.2.2.1 Reshaping into conditional tables

In [0]:
Urban_income_group = df.loc[df["City_Type"] == "Urban",["Age","Annual_Income_USD","Will_Buy_EV"]]

In [0]:
Urban_income_group.shape

In [0]:
df["City_Type"].unique()

**Create groups for suburban and rural customers.**

In [0]:
subrban_income_group = df.loc[df["City_Type"] == "Suburban",["Age","Annual_Income_USD","Will_Buy_EV"]]

In [0]:
rural_income_group = df.loc[df["City_Type"] == "Rural",["Age","Annual_Income_USD","Will_Buy_EV"]]

**Check the split of buyer preference in these groups**

In [0]:
Urban_income_group.value_counts("Will_Buy_EV")

In [0]:
Urban_income_group["Will_Buy_EV"].value_counts(normalize=True) * 100

In [0]:
Urban_income_group.describe()

In [0]:
Urban_income_group[Urban_income_group["Will_Buy_EV"] == "Yes"].describe()

In [0]:
Urban_income_group[Urban_income_group["Will_Buy_EV"] == "No"].describe()

**Observing Central Tendency across reshaped data**

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].mean().round(2) #average

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].median().round(2) #middle salary point

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].std().round(2) #square root of the variance from the mean

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].quantile(.25).round(2)

**Creating Custom dataframes and renaming columns**

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].agg(['mean', 'count']).rename(columns={'mean': 'Avg_Annual_Income_USD', 'count': 'Count_People'})


In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].agg(['mean', 'median', 'count', 'min', 'max']).rename(columns={'mean': 'Avg_Annual_Income_USD','median': 'Median_Annual_Income_USD','count': 'Count_People','min':'Min_Annual_Income_USD','max':'Max_Annual_Income_USD'})


###2.2.3 Visualizing Distributions on unscaled data for business reporting - Univariate and Multivariate

In [0]:
pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.barplot(
    data=df,
    x="City_Type",
    y="Annual_Income_USD",
    hue="Will_Buy_EV",
    estimator="mean", 
)

plt.title("Average Annual Income by City Type and EV Purchase Decision")
plt.xlabel("City Type")
plt.ylabel("Average Annual Income (USD)")

# Add data labels on each bar
for container in plt.gca().containers:
    plt.bar_label(container, fmt="%.0f", padding=3, fontsize=9)

plt.tight_layout()
plt.show()

In [0]:

plt.figure(figsize=(10, 6))

sns.histplot(data=df, x="Annual_Income_USD", hue="Will_Buy_EV", kde=True, element="step", stat="density", common_norm=False)

plt.title("Distribution of Annual Income by Will_Buy_EV")
plt.xlabel("Annual Income")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.histplot(data=df, x="Daily_Commute_km", hue="Will_Buy_EV", kde=True, element="step", stat="density", common_norm=False)

plt.title("Distribution of Daily Commute in Km by Will_Buy_EV")
plt.xlabel("Daily Commute_Km")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.countplot(data=df, x="Number_of_Cars_Owned", hue="Will_Buy_EV")

plt.title("Distribution of Number_of_Cars_Owned by Will_Buy_EV")
plt.xlabel("Number_of_Cars_Owned")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [0]:
# Select only numerical columns and compute the correlation matrix
numerical_df = df.select_dtypes(include="number")

correlation_matrix = numerical_df.corr()

correlation_matrix

In [0]:

plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=0.5)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.show()

##2.4 Transforming the data through standard scaling for machine learning

In [0]:
from sklearn.preprocessing import StandardScaler

# Select numerical columns
numerical_cols = df.select_dtypes(include="number").columns

# Apply z-score scaling (standardization)
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df_scaled[numerical_cols].head()

In [0]:
correlation_matrix_scaled = df_scaled[numerical_cols].corr()

In [0]:
plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix_scaled, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=0.5)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.show()

In [0]:
plt.figure(figsize=(10, 6))

sns.scatterplot(data=df, x="Age", y="Annual_Income_USD", hue="Will_Buy_EV")

plt.title("Scatter Plot of Age vs Annual Income")
plt.xlabel("Age")
plt.ylabel("Annual Income (USD)")

plt.tight_layout()
plt.show()

##2.5 Statistical Tests on Transformed Data